## Tutorial of running scPrediXcan protocol

#### The code to run for setting up the system
Run the following code in the terminal. For conda environment installation, if you use MacOS, select 'scPrediXcan_mac_env.yml'; if you use Linux system, select 'scPrediXcan_env.yml'.

In [1]:
# Create a folder and perform all the analysis inside it
!mkdir scTWAS # user can change the name here
!cd scTWAS

# Clone repositories for step-1
!git clone https://github.com/hakyimlab/scPrediXcan.git # Create and activate conda environment
!conda env create -f scPrediXcan/scPrediXcan_mac_env.yml # Make sure you have conda installed first

Cloning into 'scPrediXcan'...
remote: Enumerating objects: 562, done.
remote: Counting objects: 100% (66/66), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 562 (delta 30), reused 50 (delta 19), pack-reused 496 (from 1)
Receiving objects: 100% (562/562), 14.86 MiB | 9.17 MiB/s, done.
Resolving deltas: 100% (176/176), done.


Then select 'scPrediXcan' as the kernel of the jupyter notebook.

In [2]:
# git clone the repos for step-2 and step-3
!git clone https://github.com/hakyimlab/PredictDb-nextflow.git 
!git clone https://github.com/hakyimlab/MetaXcan

Cloning into 'PredictDb-nextflow'...
remote: Enumerating objects: 389, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 389 (delta 6), reused 10 (delta 5), pack-reused 370 (from 1)
Receiving objects: 100% (389/389), 156.36 KiB | 2.84 MiB/s, done.
Resolving deltas: 100% (228/228), done.
Cloning into 'MetaXcan'...
remote: Enumerating objects: 3789, done.
remote: Counting objects: 100% (648/648), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 3789 (delta 596), reused 550 (delta 548), pack-reused 3141 (from 2)
Receiving objects: 100% (3789/3789), 10.33 MiB | 6.91 MiB/s, done.
Resolving deltas: 100% (2629/2629), done.


#### Dataset preparation
In this step, firstly we will download reference gene epigenomics features file and some example files from [Zenodo](https://doi.org/10.5281/zenodo.17914799). The epigenomic feature file is required for the analysis, while the example files are provided solely to illustrate the expected input data format for each step. In real analyses, users should replace these example files with their own data (e.g., GWAS summary statistics).

In [ ]:
#Download the all the data from Zenodo, this process may take a few hours.
!zenodo_get 18124207

Title: scPrediXcan protocol datasets with files and examples
Keywords: 
Publication date: 2026-01-01
DOI: 10.5281/zenodo.18124207
Total size: 11.9 GB

File: Download_dataset.tar.gz (11.9 GB)
Link: https://zenodo.org/api/records/18124207/files/Download_dataset.tar.gz/content
100% [..............................................] 11873539065 / 1187353906516 / 1187353906580 / 1187353906516 / 1187353906572 / 1187353906500 / 1187353906592 / 1187353906552 / 1187353906576 / 1187353906592 / 1187353906564 / 1187353906500 / 1187353906508 / 1187353906568 / 1187353906580 / 1187353906592 / 1187353906528 / 1187353906528 / 1187353906516 / 1187353906516 / 11873539065
Checksum is correct for Download_dataset.tar.gz. (46b4531e1eaa1a349b8467b448c10c52)

All specified files have been processed.


In [23]:
!tar -xzf Download_dataset.tar.gz && rm Download_dataset.tar.gz

Here, we prepare the gene expression file and epigenomics file for step-1 ctPred training. Run the following code in a python script or jupyter notebook.

In [28]:
!python scPrediXcan/Scripts/ctPred/data_prep.py \
    --exp './Download_dataset/Acinar_normcounts.csv' \
    --epi './Download_dataset/Gene_epigenomics_v0.csv' \
    --output './Download_dataset/training_cell.csv'

Training data saved to ./training_cell.csv


Or you can run these code with example file paths:

In [10]:
# Import Pandas
import pandas as pd
# Load the individual-by-cell matrix data
Exp = pd.read_csv('./Download_dataset/Acinar_normcounts.csv')
# # Load the epigenomics of protein-coding genes
epi = pd.read_csv('./Download_dataset/Gene_epigenomics_v0.csv', index_col=0) # Select the genes in the epigenomic features file and fill those # genes not detected in the single-cell data with 0, otherwise it # may affect downstream percentile calculation.
selected_exp = epi.iloc[:, [0]].merge(Exp, on='gene_name', how='left').fillna(0)

# Calculate mean expression
cols_to_average = selected_exp.columns[1:] 
selected_exp['mean_expression'] = selected_exp[cols_to_average].mean(axis=1)
selected_exp = selected_exp.drop(columns=cols_to_average)
# Convert to rank-based percentiles and merge with epigenomics file 
selected_exp['mean_expression'] = selected_exp['mean_expression' ].rank(method='average', pct=True)
# Merge expression percentiles with epigenomics file 
data_training = epi.merge(selected_exp, on='gene_name',how='left') 
# Save the file and release memory 
data_training.to_csv('./Download_dataset/training_cell.csv')
# Here you can choose the path you like for the training csv file, and name it with the cell type/state.
del [selected_exp, cols_to_average, Exp, epi]

Template code where users can replace the file paths by your own:

In [ ]:
# Import Pandas
import pandas as pd
# Load the individual-by-cell matrix data
# replace the path by the real path of your data
Exp = pd.read_csv('your_expression_matrix.csv') 

# Load the epigenomics of protein-coding genes
# replace the path by the real path of your data
epi = pd.read_csv('the_epigenomics_of_genes.csv', index_col=0) 
# Select the genes in the epigenomic features file and fill those genes not detected in the single-cell data with 0, otherwise it may affect downstream percentile calculation.
selected_exp = epi.iloc[:, [0]].merge(Exp, on='gene_name', how='left').fillna(0)

# Calculate mean expression
cols_to_average = selected_exp.columns[1:] 
selected_exp['mean_expression'] = selected_exp[cols_to_average].mean(axis=1)
selected_exp = selected_exp.drop(columns=cols_to_average)
# Convert to rank-based percentiles and merge with epigenomics file
selected_exp['mean_expression'] = selected_exp['mean_expression' ].rank(method='average', pct=True)
# Merge expression percentiles with epigenomics file 
data_training = epi.merge(selected_exp, on='gene_name',how='left') 
# Save the file and release memory 
# replace the path by the real path of your data
data_training.to_csv('training_cell.csv')
# Here you can choose the path you like for the training csv file, and name it with the cell type/state.
del [selected_exp, cols_to_average, Exp, epi]

#### Make the results folder

In [11]:
# Make the folder to put the results 
!mkdir Results

#### Step1: ctPred model training
Run the following code in the terminal. In this step, we use the dataset prepared above to train the ctPred model. The ctPred model is predicting gene's expression from epigenomic features. The epigenomic features of each gene is obtained from a foundation model, Enformer. For each gene, we use the 128 * 896 bp long sequence centered at the TSS from the reference human genome sequence as the input of Enformer, and Enformer gives 896bins * 5313 epigenomic tracks as the output. The epigenomic tracks include different histone marks, transcription factor bindings, and so on. As the input of the ctPred, we use the mean of the central four bins, which is a 1 * 5313 epigenomic tracks vector as the input.

- Before starting the training, make sure to modify the JSON configuration file in the code below to specify the paths where the model weights and output figures will be saved. 
- The JSON file path is 'scPrediXcan/Scripts/ctPred_train.json', modify the "Model_path" and "fig_path". Here we use "./Results" as default, since we just create the "./Results" folder to store the outputs.
- In this step, the output should be a .pt file storing the model weights.

Script with example file paths:

In [12]:
# Train ctPred and save the results
!python scPrediXcan/Scripts/ctPred/ctPred_train.py \
    --parameters scPrediXcan/Scripts/ctPred/ctPred_train.json \
    --cell_file './Download_dataset/training_cell.csv'

Template code where users can replace the file paths by your own:

In [ ]:
# Train ctPred and save the results
!python cscPrediXcan/Scripts/ctPred/ctPred_train.py \
    --parameters cscPrediXcan/Scripts/ctPred/ctPred_train.json \
    --cell_file 'training_cell.csv' # your file

#### Predict the personalized gene expressions

Once we have the ctPred model, we can predict the personalized gene expression given the personalized epigenomic features. Here, the personalized epigenomic features are predicted by Enformer, using personalized gene DNA sequences as the input. In the tutorial, to save the time and computational resources, here we only use six example personalized epigenomic features in the './Download_dataset/Personalized_epigenomics' folder. Each .h5 file contains a central 4 bins * 5313 epigenomic features matrix as the input. The output of this step is the personalized gene expression in the './Results/Personalized_predictions.csv' file.

In [13]:
!python scPrediXcan/Scripts/ctPred/predict_h5.py \
    --model_path ./Results/training_cell_ctPred.pt \
    --input_folder Download_dataset/Personalized_epigenomics \
    --ref_csv Download_dataset/coding_gene_biomart_TSS_dictionary.csv \
    --output_path ./Results/Personalized_predictions.csv

Processing 6 samples...
Samples |████████████████████████████████████████| 6/6 (100.0%) 49.5s

Processed 117378 gene predictions
Final file framing |████████████████████████████████████████| 19563/19563 (100.0%) 59.6s

✓ Completed in 59.7s
  Output: ./Results/Personalized_predictions.csv
  Genes: 19563, Samples: 6
  Unmatched regions: 0


#### Step2: l-ctPred training
Run the following code in the terminal. In this step, we use a nextflow pipeline from MetaXcan for l-ctPred (a SNP-based elastic-net model) generation. The inputs include a genotype file, a gene annotation file, a SNP annotation file and a ctPred-predicted cell-type-specific gene expression file.
- In this step, the output should be the sqlite database and covariates file in the 'filtered_db' folder.

Script with example file paths:

In [19]:
# Run the script
!nextflow run ./PredictDb-nextflow/main.nf \
--gene_annotation ./Download_dataset/Gene_anno.txt \
--snp_annotation ./Download_dataset/T2D_snp_anno_v3.txt \
--genotype ./Download_dataset/T2D_dosage_v3.txt \
--gene_exp ./Download_dataset/Acinar_train_v0.csv \
--outdir ./Results \
--keepIntermediate \
--nfolds 3 \
-profile local \
-resume

Nextflow 25.10.2 is available - Please consider updating your version to it

 N E X T F L O W   ~  version 24.04.3

Launching `./PredictDb-nextflow/main.nf` [exotic_goldstine] DSL2 - revision: b024122f78

Run Name          : exotic_goldstine
Gene annotation   : ./Download_dataset/Gene_anno.txt
SNP annotation    : ./Download_dataset/T2D_snp_anno_v3.txt
Expression file   : ./Download_dataset/Acinar_train_v0.csv
Genotype file     : ./Download_dataset/T2D_dosage_v3.txt
Max Resources     : 480 GB memory, 48 cpus, 10d time per job
Output dir        : ./Results
Launch dir        : /Users/yichaozhou/Charles_files/project/protocol_check/New_check_full_v2
Working dir       : /Users/yichaozhou/Charles_files/project/protocol_check/New_check_full_v2/work
Script dir        : /Users/yichaozhou/Charles_files/project/protocol_check/New_check_full_v2/PredictDb-nextflow
User              : yichaozhou
----------------------------------------------------
[-        ] PREPROCESS:parse_geneAnnot     -
[-     

To reduce runtime and computational cost in this tutorial, we fit the ℓ-ctPred model using expression data from only 20 individuals and ~200 genes. As a result, this step typically completes in about 5 minutes.

Because of the small sample size and the use of random seeds in the PredictDb pipeline, this step may occasionally fail due to one or two genes having constant expression values within the same batch, where each batch only has around 6 people. In real applications, where expression data from more than 100 individuals are typically available, this issue does not occur.

Such failures are rare. If it does happen—indicated by the absence of the filtered_db folder in the ./Results directory—please run the code below and then re-run the code in the previous cell. In most cases, the step will succeed on the second attempt.

Meanwhile, we also provide the tutorial code for Step 3 that is independent of this step, which can be found in the second cell of Step 3.


In [18]:
!rm -r ./Results/pipeline_info

Template code where users can replace the file paths by your own:

In [9]:
# Replace the file paths with your own here
!nextflow run ./PredictDb-nextflow/main.nf  \
--gene_annotation 'Gene_anno.txt' \ 
--snp_annotation 'snp_annot.txt' \
--genotype 'genotype_file' \
--gene_exp 'ctPred_predicted_gene_expression.csv' \ 
--outdir results_path \
--keepIntermediate \
-profile local \
-resume

#### Step3: S-PrediXcan running
Run the following code in the terminal. In this step, we use Summary-PrediXcan(S-PrediXcan) to run the association test. The S-PrediXcan github repo with description is [here](https://github.com/hakyimlab/MetaXcan/wiki/S-PrediXcan-Command-Line-Tutorial). The input data include a transcriptome model sqlite database (i.g., l-ctPred), a GWAS/Meta Analysis summary statistics, and SNP covariance matrices. 
- In this step, the output should be a .csv file with gene-level summary statistics.

Script with newly-generated database from last step:
- Since the previous step uses a toy expression file containing only ~200 genes, the resulting models end up using 0% of SNPs(which is not a bug here). In the next cell, we show the results obtained using a database trained on ~20,000 genes, which uses ~86% of SNPs. Training ℓ-ctPred on all genes can take more than 16 hours, which is why we do not include that step directly in the tutorial.

In [20]:
!python3 ./MetaXcan/software/SPrediXcan.py \
    --model_db_path ./Results/filtered_db/predict_db_Model_training_filtered.db \
    --model_db_snp_key varID \
    --covariance ./Results/filtered_db/predict_db_Model_training_filtered.txt.gz \
    --gwas_file ./Download_dataset/T2D_harmonized.txt.gz \
    --snp_column panel_variant_id \
    --effect_allele_column effect_allele \
    --non_effect_allele_column non_effect_allele \
    --beta_column effect_size \
    --pvalue_column pvalue \
    --keep_non_rsid \
    --output_file ./Results/S_prediXcan_Acinar.csv

WARNING - Missing --gwas_h2 and --gwas_N are required to calibrate the pvalue and zscore.
INFO - Processing GWAS command line parameters
INFO - Building beta for ./Download_dataset/T2D_harmonized.txt.gz and ./Results/filtered_db/predict_db_Model_training_filtered.db
INFO - Reading input gwas with special handling: ./Download_dataset/T2D_harmonized.txt.gz
INFO - Processing input gwas
INFO - Aligning GWAS to models
INFO - Trimming output
INFO - Successfully parsed input gwas in 10.683869041968137 seconds
INFO - Started metaxcan process
INFO - Loading model from: ./Results/filtered_db/predict_db_Model_training_filtered.db
INFO - Loading covariance data from: ./Results/filtered_db/predict_db_Model_training_filtered.txt.gz
INFO - Processing loaded gwas
INFO - Started metaxcan association
INFO - 0 % of model's snps used
WARNING - IMPORTANT: The pvalue and zscore are uncalibrated for inflation
INFO - Sucessfully processed metaxcan association in 0.09669895796105266 seconds


For the warning ‘Missing --gwas_h2 and --gwas_N are required to calibrate the p-value and z-score’, please refer to the [documentation](https://github.com/hakyimlab/MetaXcan/wiki/S-PrediXcan-phi-inflation-correction). It describes an update to the pipeline introduced by my colleague, which adds an option to calibrate potential p-value inflation. However, the pipeline can still be executed without this step or the required inputs (i.e., --gwas_h2 and --gwas_N), as shown above.

Script with example file paths:

In [21]:
!python3 ./MetaXcan/software/SPrediXcan.py \
    --model_db_path ./Download_dataset/scPrediXcan_Beta_T2D.db \
    --model_db_snp_key varID \
    --covariance ./Download_dataset/predict_db_Model_training_filtered.txt \
    --gwas_file ./Download_dataset/T2D_harmonized.txt.gz \
    --snp_column panel_variant_id \
    --effect_allele_column effect_allele \
    --non_effect_allele_column non_effect_allele \
    --beta_column effect_size \
    --pvalue_column pvalue \
    --gwas_N 492191 \
    --gwas_h2 0.1083 \
    --keep_non_rsid \
    --output_file ./Results/S_prediXcan_Beta.csv

INFO - Processing GWAS command line parameters
INFO - Building beta for ./Download_dataset/T2D_harmonized.txt.gz and ./Download_dataset/scPrediXcan_Beta_T2D.db
INFO - Reading input gwas with special handling: ./Download_dataset/T2D_harmonized.txt.gz
INFO - Processing input gwas
INFO - Aligning GWAS to models
INFO - Trimming output
INFO - Successfully parsed input gwas in 28.150978541933 seconds
INFO - Started metaxcan process
INFO - Loading model from: ./Download_dataset/scPrediXcan_Beta_T2D.db
INFO - Loading covariance data from: ./Download_dataset/predict_db_Model_training_filtered.txt
INFO - Processing loaded gwas
INFO - Started metaxcan association
INFO - 10 % of model's snps found so far in the gwas study
INFO - 20 % of model's snps found so far in the gwas study
INFO - 30 % of model's snps found so far in the gwas study
INFO - 40 % of model's snps found so far in the gwas study
INFO - 50 % of model's snps found so far in the gwas study
INFO - 60 % of model's snps found so far in 

Template code where users can replace the file paths by your own:

In [ ]:
# Run S-PrediXcan using ctPred models and GWAS summary statistics and replace the folder path with your own.
!python3 ./MetaXcan/software/SPrediXcan.py \
    --model_db_path 'l-ctPred_celli.db' \
    --model_db_snp_key varID \
    --covariance 'covariance.txt.gz' \
    --gwas_folder data/GWAS \
    --gwas_file_pattern ".*gz" \
    --snp_column panel_variant_id  \
    --effect_allele_column effect_allele \
    --non_effect_allele_column non_effect_allele \
    --beta_column effect_size \
    --pvalue_column pvalue \
    --gwas_N N \
    --gwas_h2 h2 \
    --keep_non_rsid \
    --output_file './Results/TWAS_result.csv'